# 07 — Recommendation Quality Evaluation

This notebook is the core of our **Qualitative Evaluation Phase**. 

## Goal
Systematically stress-test the `HybridEngine` to identify:
- **Semantic Drift:** Do results stay on topic?
- **Repetition/Collapse:** Are the top results too similar?
- **Metadata Leakage:** Do low-quality/spammy podcasts appear?
- **Ranking Realism:** Do confidence scores feel psychologically believable?
- **Explanation Quality:** Do the reasons given make sense for the result?

## Methodology
We use the live `HybridEngine` and run it against a library of diverse queries. We manually inspect the top 10 results for each, focusing on the *perceived intelligence* of the system.

In [ ]:
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import warnings

# Ensure backend is discoverable
sys.path.append(os.path.abspath(".."))

from backend.engines.hybrid_engine import HybridEngine
from backend.utils.loader import artifacts

warnings.filterwarnings('ignore')

print("Initializing Hybrid Engine...")
artifacts.load_all()
engine = HybridEngine()
print("Engine Ready.")

In [ ]:
def evaluate_query(query, limit=10, preferred_categories=None):
    """
    Runs a query through the hybrid engine and displays detailed evaluation metadata.
    """
    print(f"\n{'='*80}")
    print(f"EVALUATING QUERY: '{query}'")
    if preferred_categories:
        print(f"USER PREFERENCES: {preferred_categories}")
    print(f"{'='*80}\n")
    
    results = engine.recommend(
        query=query, 
        limit=limit, 
        preferred_categories=preferred_categories
    )
    
    eval_data = []
    for i, res in enumerate(results):
        # Find full metadata from artifacts for deep inspection
        # We search by title since we don't have a direct ID-to-meta lookup in artifacts.metadata (it's a list)
        full_meta = next((m for m in artifacts.metadata if m['podcast_id'] == res.podcast_id), {})
        
        eval_data.append({
            'Rank': i + 1,
            'Score': f"{res.blended_score*100:.1f}%",
            'Title': res.title,
            'Author': res.author,
            'Categories': res.categories,
            'Explanation': res.explanation,
            'Semantic': f"{res.semantic_score:.2f}",
            'Collab': f"{res.collaborative_score:.2f}",
            'Description Snippet': full_meta.get('description', '')[:100] + "..."
        })
    
    df = pd.DataFrame(eval_data)
    return df

## TEST 1 — Technology & AI (Precision Check)
**Goal:** Does it return high-quality tech podcasts or generic/spam titles?

In [ ]:
evaluate_query("Artificial Intelligence and future of society")

## TEST 2 — Mixed Interests (Personalization & Boosting Check)
**Goal:** How well does the `preferred_categories` boost influence the ranking?

In [ ]:
evaluate_query("Modern productivity", preferred_categories=["Technology", "Business"])

## TEST 3 — Niche Exploration (Semantic Drift Check)
**Goal:** Does a specific query like "Ancient Rome" stay in history/archaeology or drift into random society podcasts?

In [ ]:
evaluate_query("Ancient Rome and Archaeology")

## TEST 4 — Diversity & Repetition (Collapse Check)
**Goal:** Does it return 10 nearly identical shows, or a diverse mix of related sub-topics?

In [ ]:
evaluate_query("Business startups and entrepreneurship")

## TEST 5 — Quality Penalty Check (Spam/Weak Metadata)
**Goal:** Look for generic titles like "Podcast", "Test", or entries with very short descriptions. Are they correctly penalized below higher-quality content?

In [ ]:
evaluate_query("interesting stories")

## Final Analysis & Improvements (May 2026)

Following systematic evaluation, the `HybridEngine` was refined with the following improvements:

1. **Semantic Drift Protection:** Increased threshold for semantic relevance. Results with `norm_semantic_score < 0.25` are now aggressively penalized to prevent off-topic results in search.
2. **Diversity Enhancement:** Increased category and author redundancy penalties (0.05 and 0.04 respectively) to ensure a more varied top 10.
3. **Metadata Hygiene:** Added specific penalties for URL-like titles and generic "General" categorizations.
4. **Intelligent Explanations:** Variety increased from a single robotic template to category-aware and confidence-aware explanations.

### Automated Suite
For repeatable testing, run:
```bash
python scripts/eval_recommendations.py
```
This generates a markdown report in `docs/eval_report.md`.